In [2]:

import numpy as np
from time import time
from scipy.optimize import minimize, OptimizeResult, basinhopping

from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator


from qopt_best_practices.sat_mapping import SATMapper

from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian




filename = 'test_N2_W2'
p: int = 1
init_type = 'random'

seed = 1
rng = np.random.default_rng()

def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')
    

data_file = f'/lustre/scratch127/qpg/jc59/out/oriented/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)


backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single'
)
backend = AerSimulator(**backend_options)
backend.set_option("n_qubits", hamiltonian.num_qubits)

qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = p,
    flatten=True
)
transpiled_qc = transpile(qc, backend, optimization_level=3, seed_transpiler=seed)
print_circuit_info(transpiled_qc, '(Transpiled) Circuit')



graph = circuit_to_graph(qc, qc.parameters[p])

swap_strat = SwapStrategy.from_line(range(graph.order()))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat
)
if remapped_g is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

circ_dict = circuit_construction(singles, doubles, None, swap_strat, edge_coloring, {}, p)

circuit = circ_dict["circuit_to_sample"]
circuit.remove_final_measurements()
print_circuit_info(circuit, '(Transpiled) Remapped Circuit')

backend = AerSimulator(**backend_options)

qaoa_depth = len(circuit.parameters) // 2


if init_type == 'ramp':
    t = 0.7 * p
    betas = np.linspace(
        (1 / p) * (t * (1 - 0.5 / p)), (1 / p) * (t * 0.5 / p), p
    )
    gammas = betas[::-1]
    init_params = betas.tolist() + gammas.tolist()
elif init_type == 'warm':
    if p == 4:
        init_params = [ 9.33323444e-01,  7.08009649e-03,  7.36344025e-01,  9.37923754e-01,
            2.29973290e-02,  2.75044958e-03,  8.34971625e-04, -2.92071056e-04]
    elif p == 2:
        init_params = [0.86782694, 0.99261561, 0.02069131, 0.84516142]
    else:
        raise Exception('No warm values available')
    print('Using warm init values')
else:
    init_params = rng.uniform(0, 1, qaoa_depth).tolist() + rng.uniform(0, 1, qaoa_depth).tolist()
print(f'Init: {init_params}')


history = []
best_func_val = np.inf
best_params = init_params
best_sv = np.array([])

def callback(intermediate_result: OptimizeResult):
    print(f'Current params: {intermediate_result.x}. Current func value: {intermediate_result.fun}')
    

def callback_cobyla(xk: np.ndarray):
    print(f'Current params: {xk}.')

def callback_basinhopping(x: np.ndarray, f: float, accept: bool):
    print(f'Current params: {x}. Current func value: {f}')


def cvar(energies, alpha=1.0):
    sorted_energies = sorted(energies)
    end_idx = int(alpha * len(energies))
    return np.sum(sorted_energies[0:end_idx]) / end_idx

int_samples = [np.array([int(x) for x in np.binary_repr(y, width=hamiltonian.num_qubits)]) for y in range(2**hamiltonian.num_qubits)]
evals = np.array([
    sample @ Q @ sample for sample in int_samples
]) + offset


def objective(x: np.ndarray):
    start = time()
    assigned_circuit = circuit.assign_parameters(x, inplace=False)
    assigned_circuit.save_statevector()
    job = backend.run([assigned_circuit])
    result = job.result()
    data = result.results[0].data
    sv = np.asarray(data.statevector)
    sampling_time = time() - start
    start = time()
    energy = np.sum(np.abs(sv) ** 2 * evals)
    
    global best_func_val
    global best_params
    global best_sv
    if energy < best_func_val:
        best_func_val = energy
        best_params = x
        best_sv = sv

    classical_post_process_time = time() - start
    history.append((sampling_time, energy, x.tolist(), sv, classical_post_process_time))
    return energy


# method = "COBYLA"
# result = minimize(
#     objective, x0=init_params, 
#     method=method, 
#     bounds=tuple((0,1) for _ in range(2 * p)), 
#     options={"maxiter": 10000, "maxfev": 10000, "rhobeg": 0.01,},  #  "ftol": 1e-7
#     callback=callback if method not in ['SLSQP', 'COBYLA', 'TNC'] else callback_cobyla
# )
# print(result)




(Transpiled) Circuit has 38 2Q gates     and 17 2Q depth
(Transpiled) Remapped Circuit has 124 2Q gates     and 28 2Q depth
Init: [0.4488035882786181, 0.45800550402114226]


In [3]:
energy = objective(np.array(init_params))

In [4]:
energy

np.float64(119.73805540014114)

In [5]:
history

[(0.008535623550415039,
  np.float64(119.73805540014114),
  [0.4488035882786181, 0.45800550402114226],
  array([-0.07019395+0.02848469j, -0.01506618+0.0004072j ,
          0.00853737-0.01155425j, ...,  0.00172525-0.00166086j,
         -0.01757933+0.01966788j, -0.00989784+0.00284004j], shape=(1024,)),
  0.0002493858337402344)]

In [6]:
evals

array([ 22.,  12.,  11., ..., 388., 438., 508.], shape=(1024,))

In [9]:
np.nonzero(evals == 0)

(array([ 72, 516]),)

In [11]:
np.binary_repr(72, 10)

'0001001000'

In [13]:
np.binary_repr(516, 10)

'1000000100'

In [15]:
np.abs(best_sv[72]) ** 2

np.float64(0.0006295952022241923)

In [16]:
np.abs(best_sv[516]) ** 2

np.float64(0.0005326337631178625)

In [17]:
transpiled_qc.draw(fold=-1)

┌───┐┌──────────────────┐                                                                                                                                                                                                                                       ┌────────────┐                                                                                                                                                                                                                                                                                                                                   
q_0: ┤ H ├┤ Rz((-39.5)*γ[0]) ├─■─────────────■───────────────────────────■───────────────────────────■─────────────────────────────────────────■───────────────────────────────────────────■──────────────────────────────────────────────────────────■──────────────┤ Rx(2*β[0]) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├───┤└┬────────────────┬┘ │ZZ(11*γ[0])  │                           │                           │                                         │                                           │                                                          │              └────────────┘                                                                             ┌────────────┐                                                                                                                                                                                                                                        
q_1: ┤ H ├─┤ Rz((-42)*γ[0]) ├──■─────────────┼─────────────■─────────────┼─────────────■─────────────┼───────────────────────────■─────────────┼────────────────────────────■──────────────┼───────────────────────────────────────────■──────────────┼────────────────────────────────────────────■─────────────────────────────────────────────■──────────────┤ Rx(2*β[0]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├───┤ ├────────────────┤                │ZZ(10*γ[0])  │ZZ(10*γ[0])  │             │             │                           │             │                            │              │                                           │              │                                            │                                             │              └────────────┘                               ┌────────────┐                                                                                                                                                                                           
q_2: ┤ H ├─┤ Rz((-42)*γ[0]) ├────────────────■─────────────■─────────────┼─────────────┼─────────────┼─────────────■─────────────┼─────────────┼──────────────■─────────────┼──────────────┼────────────────────────────■──────────────┼──────────────┼─────────────────────────────■──────────────┼──────────────────────────────■──────────────┼─────────────────────────────────────────────■─────────────┤ Rx(2*β[0]) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├───┤┌┴────────────────┴┐                                           │ZZ(10*γ[0])  │ZZ(10*γ[0])  │             │ZZ(11*γ[0])  │             │              │             │              │                            │              │              │                             │              │                              │              │                                             │             └────────────┘